# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/imatiq/ML_Internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/imatiq/ML_Internship"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")


Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

**Fields checked:** `impressions_90d`, `clicks_90d`, `sessions_90d`, `ctr`, `avg_position`, `word_count`.

Traffic metrics are heavy-tailed, as expected for web data. `impressions_90d` has a mean of 5,200 but a median of only 731 — the mean sits 7.1x above the median — and the top 1% of pages by impressions account for roughly **25% of all impressions** in the dataset. `clicks_90d` (mean/median ≈ 16x) and `ctr` (≈7x, driven by a handful of tiny-denominator pages) are even more skewed. `avg_position` and `word_count` are much closer to symmetric (word_count mean/median ≈ 1.1x).

Because of this, plain correlation on raw impressions/clicks/CTR would be dominated by a few giant pages. For the signal tests below I use `log1p()` on traffic counts, Spearman (rank) correlation, or volume-weighted rates instead of raw Pearson/means.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd, numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)

for col in ["impressions_90d", "clicks_90d", "sessions_90d", "word_count", "avg_position", "ctr"]:
    s = df[col].dropna()
    print(f"{col:16s} n={len(s):6d}  mean={s.mean():10.2f}  median={s.median():10.2f}  "
          f"p90={s.quantile(.9):10.2f}  max={s.max():10.2f}  mean/median={s.mean()/max(s.median(),1e-9):.1f}x")

top1pct = df.nlargest(int(len(df) * 0.01), "impressions_90d")
print(f"\nTop 1% of pages by impressions_90d hold "
      f"{top1pct['impressions_90d'].sum() / df['impressions_90d'].sum():.1%} of all impressions.")


impressions_90d  n= 30000  mean=   5200.37  median=    731.00  p90=  12136.40  max= 517715.00  mean/median=7.1x
clicks_90d       n= 30000  mean=     16.10  median=      1.00  p90=     32.00  max=   4178.00  mean/median=16.1x
sessions_90d     n= 30000  mean=     37.07  median=      7.00  p90=     88.00  max=   4345.00  mean/median=5.3x
word_count       n= 22301  mean=   3107.76  median=   2877.00  p90=   5327.00  max=   9546.00  mean/median=1.1x
avg_position     n= 30000  mean=     16.34  median=     10.80  p90=     36.80  max=    245.00  mean/median=1.5x
ctr              n= 30000  mean=      0.51  median=      0.07  p90=      0.65  max=    100.00  mean/median=7.3x

Top 1% of pages by impressions_90d hold 24.9% of all impressions.


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

### Test 1 — "Longer pages get more impressions"
**Test:** Spearman correlation of `word_count` vs `impressions_90d` (n=22,301; 7,699 rows have no word count and are excluded), plus median impressions by `word_count_tier`.
**Result:** rho = 0.299 (p < 0.001). Median impressions rise monotonically across tiers: `<1000`→4, `1000-2000`→172, `2000-3500`→997, `3500+`→1,340.
**Verdict: CONFIRMED.** Direction holds, monotonic across tiers, comfortably above the sample floor — but the correlation is weak-to-moderate (0.3), so word count alone explains only part of the impressions story.

### Test 2 — "Pages ranking higher (lower avg_position) get higher CTR"
**Test:** restrict to `avg_position > 0` (drops 1,205 no-data rows). Compare median CTR vs **volume-weighted** CTR (`total clicks / total impressions`) per `position_tier`.
**Result:** median CTR is *not* monotonic — `top_3` shows a median of 0.00%, worse-looking than `page_1` or `striking`. That's a volume-floor artifact: `top_3`'s median page only gets ~53 impressions/90d, so one extra click swings its CTR by ~1.9 points (matches the warning in the data dictionary). The weighted CTR tells the real story: `top_3`=0.49% → `page_1`=0.35% → `striking`=0.35% → `page_3_5`=0.15% → `deep`=0.04% — a clean decline as position worsens.
**Verdict: CONFIRMED — but only once read correctly.** The naive per-row median for this signal is actively misleading at low-volume tiers; the aggregate (weighted) rate is what should be trusted and reported.

### Test 3 — "Content updated more recently is less likely to be declining"
**Test:** `is_declining_label` rate by `freshness_tier`.
**Result:** `0-30d`→51.1% (n=20,480), `31-90d`→58.9% (n=175), `91-180d`→61.1% (n=9,171), `181+d`→47.1% (n=174). No monotonic "fresher = safer" pattern: the middle bucket has the *highest* decline rate, and the oldest bucket has the *lowest*.
**Verdict: MIXED.** The two largest buckets (0-30 vs 91-180) point the expected direction (newer content declines less), but the smallest bucket (`181+`, n=174 — thin, near the reliability edge) breaks the pattern entirely. Freshness alone isn't a clean predictor of the decline label in this slice.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Test 1
sub = df.dropna(subset=["word_count"])
from scipy.stats import spearmanr
rho, p = spearmanr(sub["word_count"], sub["impressions_90d"])
print("Test 1 — word_count vs impressions_90d: n=%d, rho=%.3f, p=%.2e" % (len(sub), rho, p))
print(sub.groupby("word_count_tier")["impressions_90d"].median()
      .reindex(["<1000", "1000-2000", "2000-3500", "3500+"]))

# Test 2
sub2 = df[df["avg_position"] > 0]
tbl = sub2.groupby("position_tier").agg(
    n=("ctr", "count"),
    median_ctr=("ctr", "median"),
    total_clicks=("clicks_90d", "sum"),
    total_impressions=("impressions_90d", "sum"),
)
tbl["weighted_ctr_pct"] = tbl["total_clicks"] / tbl["total_impressions"] * 100
print("\nTest 2 — CTR by position tier:")
print(tbl[["n", "median_ctr", "weighted_ctr_pct"]].reindex(["top_3", "page_1", "striking", "page_3_5", "deep"]))

# Test 3
tbl3 = df.groupby("freshness_tier")["is_declining_label"].agg(["mean", "count"]).reindex(["0-30", "31-90", "91-180", "181+"])
print("\nTest 3 — decline rate by freshness tier:")
print(tbl3)


Test 1 — word_count vs impressions_90d: n=22301, rho=0.299, p=0.00e+00
word_count_tier
<1000           4.0
1000-2000     172.0
2000-3500     997.0
3500+        1340.0
Name: impressions_90d, dtype: float64

Test 2 — CTR by position tier:
                   n  median_ctr  weighted_ctr_pct
position_tier                                     
top_3           1116        0.00          0.488544
page_1         11814        0.16          0.350324
striking        7304        0.11          0.346876
page_3_5        7242        0.03          0.154905
deep            1319        0.00          0.041359

Test 3 — decline rate by freshness tier:
                    mean  count
freshness_tier                 
0-30            0.511377  20480
31-90           0.588571    175
91-180          0.611057   9171
181+            0.471264    174


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

**Flag chosen:** `stale_visible_page` from the baseline rule (`scripts/02_baseline_score.py`) — it fires when `days_since_last_update >= 180` **and** `impressions_90d >= 500`, assuming stale-but-still-visible pages are worth flagging for refresh (i.e., more likely to be declining).

**Test:** decline rate for pages meeting both conditions vs. the dataset baseline, then decomposed into each condition on its own.

**Result:** only **17 rows** in the whole 30,000-row slice satisfy both conditions at once — below the ~50-row floor, so this specific bucket can't support a verdict by itself (its 94.1% decline rate vs. the 54.2% baseline is suggestive, not confirmable). Decomposing the two conditions separately, at full sample size:
- **Staleness alone** (`days_since_last_update >= 180`, n=174): decline rate **47.1%** — *lower* than the 54.2% baseline, the opposite of what the rule assumes.
- **Visibility alone** (`impressions_90d >= 500`, n=16,726): decline rate **59.6%** vs. 47.5% for low-visibility pages — this half of the rule does hold up.

**Verdict: INSUFFICIENT DATA** on the exact flagged bucket (n=17 < floor). But the decomposition suggests the flag's real predictive power comes from its *visibility* condition — staleness isn't independently associated with decline in this slice at all, and may be riding along on the rule's coattails rather than adding signal.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
stale = df["days_since_last_update"] >= 180
visible = df["impressions_90d"] >= 500
flag = stale & visible

print("stale_visible_page flag: n=%d, decline rate=%.3f" % (flag.sum(), df.loc[flag, "is_declining_label"].mean()))
print("Dataset baseline decline rate: %.3f" % df["is_declining_label"].mean())
print()
print("Staleness alone: n=%d, decline rate=%.3f" % (stale.sum(), df.loc[stale, "is_declining_label"].mean()))
print("Visibility alone: n=%d, decline rate=%.3f" % (visible.sum(), df.loc[visible, "is_declining_label"].mean()))
print("Low visibility: n=%d, decline rate=%.3f" % ((~visible).sum(), df.loc[~visible, "is_declining_label"].mean()))


stale_visible_page flag: n=17, decline rate=0.941
Dataset baseline decline rate: 0.542

Staleness alone: n=174, decline rate=0.471
Visibility alone: n=16726, decline rate=0.596
Low visibility: n=13274, decline rate=0.475


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

Word count and position both track outcomes in the expected direction, but CTR-by-position only shows the correct story when weighted by volume — a dashboard that just averages CTR per rank bucket will be actively misled by low-traffic top-3 pages. The `stale_visible_page` refresh rule doesn't have enough matching pages in this slice to confirm on its own, and what evidence there is suggests its visibility half is carrying the signal while the staleness half isn't independently predictive — so refresh priority should probably weight visibility over update-recency until this is re-checked on the full warehouse release, where the matching bucket will be far larger than 17 rows.

In [5]:
# No further computation needed for this section — reasoning only, backed by cells above.
print("See sections 1-3 for the numbers this reasoning is based on.")


See sections 1-3 for the numbers this reasoning is based on.


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.